# Timme et al. (2016) on CRCNS hc-8: complexity under spike swapping on more recordings

Paper: Timme et al. (2016), Front. Physiol. 7:425, doi:10.3389/fphys.2016.00425; claims file
`claims/timme-2016-criticality.yaml` (version of record, PMC5037237, with the supplement). Dataset:
CRCNS hc-8, doi:10.6080/K0PC308P, `hc-8/live/Data.zip` (MD5 `bd4e2a8685df3e55e1a3c04fbe09d9ce` in the
data set's `checksums.md5`), version `unversioned`: CRCNS data sets carry no versions and this is the
only published copy (see `culture_criticality.ipynb`).

`complexity_culture.ipynb` decides fig12a-swap-preserves-complexity on 40 recordings, because it
computes the complexity of all eight conditions. The verifier of the second round asked for the
real and swapped conditions alone on more recordings. This notebook computes them, with the same
method and generators, on 200 recordings: the 40 of `complexity_culture.ipynb` (their values are
the same here) and 160 more drawn with a fixed seed. Each recording takes about 12 s of wall time
with 4 processes, so all 435 would take about 85 min, over the 60 min limit of a notebook.

| claim | here |
|---|---|
| fig12a-swap-preserves-complexity | tested on more recordings (the outcome is recorded from `complexity_culture.ipynb`; a change is proposed from here) |

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")
import time, resource
import multiprocessing as mp
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import hc8

SEED = 20161          # the same seed as the other notebooks, so the swapped data are the same
N_SUBSETS = 100       # Methods 2.9: "the lesser of either all possible permutations of neurons or 100"
SUBSET = int(os.environ.get("HC8_SUBSET", "0"))   # only while developing
names = hc8.recording_names()
idx40 = sorted(np.random.default_rng([SEED, 0]).choice(len(names), size=40, replace=False).tolist())
rest = np.setdiff1d(np.arange(len(names)), idx40)
idx = sorted(idx40 + np.random.default_rng([SEED, 1]).choice(rest, size=160, replace=False).tolist())
if SUBSET:
    idx = idx[::max(1, len(idx) // SUBSET)][:SUBSET]
t0 = time.time()
with mp.get_context("spawn").Pool(4) as pool:
    recs = list(pool.imap(hc8.complexity_swap_record, [(i, names[i], SEED, N_SUBSETS) for i in idx], chunksize=1))
sw = pd.DataFrame(recs)
sw["in_40"] = [i in idx40 for i in idx]
print(f"{len(sw)} recordings in {(time.time() - t0) / 60:.1f} min; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
print("recordings:", len(sw), "| of them in complexity_culture.ipynb:", int(sw.in_40.sum()),
      "| cultures:", sw.culture.nunique(), "| neurons analysed:", int(sw.n_neurons.sum()))
print("swap leaves A(t) unchanged in every recording:", bool(sw.swap_act_identical.all()))
incl_a = (sw.n_size_cut_values >= 1) & (sw.n_dur_cut_values >= 1) & (sw.n_profiles >= 2)
print("recordings kept under exclusion reading (a):", int(incl_a.sum()), "of", len(sw))
for k in ("real", "swap"):
    c = sw[f"C_{k}"]
    print(f"{k:5s} complexity (bits/neuron) median {c.median():.5f}, range {c.min():.5f} to {c.max():.5f}; "
          f"subset sizes kept by the cut: median {sw[f'kkept_{k}'].median():.0f} of {sw[f'N_{k}'].median():.0f}; NaN {int(c.isna().sum())}")

## fig12a-swap-preserves-complexity

> Culture complexity does not change substantially under spike swapping, and real and swapped
> complexity are strongly correlated with most recordings near the equality line.

The test of `complexity_culture.ipynb`: rank sum test of real against swapped complexity across
recordings; the false discovery rate correction there runs over the seven randomizations of the
panel, which are not computed here, so the uncorrected p is the most significant the corrected
one could be. Strong correlation read as Pearson r >= 0.7. The 40 recordings of
`complexity_culture.ipynb` and the 160 new ones are also shown apart.

In [ ]:
def swap_test(rows, prefix):
    a, b = sw[f"{prefix}_real"][rows], sw[f"{prefix}_swap"][rows]
    ok = a.notna() & b.notna()
    a, b = a[ok], b[ok]
    p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    pr = stats.pearsonr(a, b); sr = stats.spearmanr(a, b); w = stats.wilcoxon(a, b)
    return dict(n=int(ok.sum()), real_median=a.median(), swap_median=b.median(),
                rel_change=(b.median() - a.median()) / a.median(), rank_sum_p=p,
                pearson_r=pr.statistic, spearman=sr.statistic, signed_rank_p=w.pvalue,
                median_rel_change=((b - a) / a).median())

rows = []
for prefix, lab in (("C", "complexity.m"), ("Cuncut", "variant no cut"), ("Craw", "variant no correction")):
    for sel_lab, sel in (("all", incl_a), ("the 40", incl_a & sw.in_40), ("the 160 new", incl_a & ~sw.in_40)):
        rows.append(dict(complexity=lab, recordings=sel_lab, **swap_test(sel, prefix)))
res = pd.DataFrame(rows).set_index(["complexity", "recordings"])
print(res.to_string(float_format=lambda v: f"{v:.3g}"))
t = res.loc[("complexity.m", "all")]
print(f"\ncomplexity.m, all recordings: rank sum p {t.rank_sum_p:.3g} (no change at 0.05: {t.rank_sum_p >= 0.05}); "
      f"Pearson r {t.pearson_r:.3f} (>= 0.7: {t.pearson_r >= 0.7})")
# how r depends on the largest values: r without the recordings above the 95th percentile of either
a, b = sw.C_real[incl_a], sw.C_swap[incl_a]
q = max(a.quantile(0.95), b.quantile(0.95))
k = (a <= q) & (b <= q)
print(f"Pearson r without the recordings above the 95th percentile ({int((~k).sum())} dropped): {stats.pearsonr(a[k], b[k]).statistic:.3f}; "
      f"on log10 values (both > 0, n {int(((a > 0) & (b > 0)).sum())}): "
      f"{stats.pearsonr(np.log10(a[(a > 0) & (b > 0)]), np.log10(b[(a > 0) & (b > 0)])).statistic:.3f}")
print("per culture, median per-recording relative change:",
      ((b - a) / a).groupby(sw.culture[incl_a]).median().round(2).to_dict())
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
ax[0].scatter(a, b, s=4); m = max(a.max(), b.max()); ax[0].plot([0, m], [0, m], "k--", lw=.8)
ax[0].set_xlabel("real"); ax[0].set_ylabel("swap"); ax[0].set_title("complexity.m (bits/neuron)")
ax[1].scatter(a, b, s=4); ax[1].set_xscale("log"); ax[1].set_yscale("log"); ax[1].set_xlabel("real"); ax[1].set_title("log axes")
plt.tight_layout(); plt.show()
print(f"plotted range: {min(a.min(), b.min()):.5f} to {max(a.max(), b.max()):.5f}; points <= 0 left off the log axes: {int(((a <= 0) | (b <= 0)).sum())}")

**Result: no change, as the paper says; the correlation is weaker than on the 40 recordings.** On 200 recordings the rank sum test finds no difference between real and swapped complexity (medians 0.00184 and 0.00188, uncorrected p 0.933, so the corrected p is at least that), the outcome rule of the criterion. Pearson r is 0.658, below the 0.7 reading of strong correlation (Spearman 0.539); on the 160 new recordings it is 0.598, against 0.775 on the 40 of `complexity_culture.ipynb`, whose values are reproduced exactly here. Without the 11 recordings above the 95th percentile r is 0.494. The median per-recording relative change is -0.013, but per culture it ranges from -0.91 to 2.57. Without the cut r is 0.971 and without the correction 0.997, though the paired signed-rank test then finds small decreases (p 0.000782 and 0.000421, median relative changes -0.023 and -0.007).

## Summary

| claim | outcome |
|---|---|
| fig12a-swap-preserves-complexity | reproduced (recorded from `complexity_culture.ipynb`; on 200 recordings the rank sum test agrees and Pearson r is 0.658) |

Differences from the paper's method: 200 of the 435 recordings, the real and swapped data only.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.